# CNN & LSTM Hyperparameter Tuning

This notebook is the reference tuning discipline that MLP and Transformers should match.

Main fairness rule:
- fixed CNN/LSTM architectures
- same data split and trainer
- bounded small grid as MLP and Transformers
- model selection on clean validation score only
- 3-seed confirmation after choosing the best candidate

The notebook also fixes the final-config dictionary construction bug from the previous version.

Import policy: this notebook imports the original project modules under `src/models/` and does not depend on any `_patched` filenames or patched class aliases.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find project root containing 'src'. Run this notebook from inside the project repo.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) in sys.path:
    sys.path.remove(str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT =", PROJECT_ROOT)


PROJECT_ROOT = /storage/ice1/7/5/kw53/cs7643/tmp/CS7643-Project


In [2]:
import random
import numpy as np
import torch

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Keep deterministic behavior when possible.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


In [3]:
import pandas as pd
import torch

from src.data_prep import prepare_uji_data
from src.models.mlp_coordinates import CoordinateMLPModel
from src.models.mlp_joint import JointMLPModel
from src.models.mlp_multitask import MultiTaskMLPModel
from src.models.cnn_coordinates import CNNCoordinateModel
from src.models.cnn_joint import CNNJointModel
from src.models.cnn_multitask import CNNMultiTaskModel
from src.models.lstm_coordinates import LSTMCoordinateModel
from src.models.lstm_joint import LSTMJointModel
from src.models.lstm_multitask import LSTMMultiTaskModel
from src.training import TrainConfig, train_from_tensors

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)


In [4]:
bundle = prepare_uji_data()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

joint_y_train, joint_y_val = bundle.get_targets(["joint"])
mt_y_train, mt_y_val = bundle.get_targets(["building", "floor"])
coord_y_train, coord_y_val = bundle.get_targets(["longitude", "latitude"])

in_dim = bundle.X_train.shape[1]

print("device:", device)
print("X train/val:", bundle.X_train.shape, bundle.X_val.shape)
print("coordinate_std:", bundle.coordinate_std)


device: cuda
X train/val: (19937, 1040) (1111, 1040)
coordinate_std: [123.39891  66.94215]


In [5]:
def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


## Parameter-count sanity check


In [6]:
param_rows = [
    {"model": "mlp_joint", "params": count_trainable_params(JointMLPModel(in_dim=in_dim))},
    {"model": "mlp_multitask", "params": count_trainable_params(MultiTaskMLPModel(in_dim=in_dim))},
    {"model": "mlp_coordinate", "params": count_trainable_params(CoordinateMLPModel(in_dim=in_dim, coordinate_std=bundle.coordinate_std))},

    {"model": "cnn_joint", "params": count_trainable_params(CNNJointModel(in_dim=in_dim))},
    {"model": "cnn_multitask", "params": count_trainable_params(CNNMultiTaskModel(in_dim=in_dim))},
    {"model": "cnn_coordinate", "params": count_trainable_params(CNNCoordinateModel(in_dim=in_dim, coordinate_std=bundle.coordinate_std))},

    {"model": "lstm_joint", "params": count_trainable_params(LSTMJointModel(in_dim=in_dim))},
    {"model": "lstm_multitask", "params": count_trainable_params(LSTMMultiTaskModel(in_dim=in_dim))},
    {"model": "lstm_coordinate", "params": count_trainable_params(LSTMCoordinateModel(in_dim=in_dim, coordinate_std=bundle.coordinate_std))},
]

param_df = pd.DataFrame(param_rows)
param_df["params_millions"] = param_df["params"] / 1_000_000
param_df


,model,params,params_millions
0,mlp_joint,1729037,1.729037
1,mlp_multitask,1727752,1.727752
2,mlp_coordinate,1726210,1.726210
3,cnn_joint,1584909,1.584909
4,cnn_multitask,1583624,1.583624
5,cnn_coordinate,1582082,1.582082
6,lstm_joint,1642030,1.642030
7,lstm_multitask,1640745,1.640745
8,lstm_coordinate,1639203,1.639203


## Shared helpers


In [7]:
def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def make_cfg(
    *,
    lr=2e-3,
    weight_decay=1e-4,
    batch_size=256,
    val_batch_size=512,
    max_epochs=30,
    patience=5,
    print_every=5,
    grad_clip_norm=None,
    run_name=None,
):
    return TrainConfig(
        lr=lr,
        weight_decay=weight_decay,
        batch_size=batch_size,
        val_batch_size=val_batch_size,
        max_epochs=max_epochs,
        patience=patience,
        print_every=print_every,
        grad_clip_norm=grad_clip_norm,
        run_name=run_name,
    )

def result_row(name, result):
    row = {"model": name, "best_epoch": result.best_epoch}
    row.update(result.best_metrics)
    return row

def run_trial(model, y_train, y_val, cfg, *, family, task, seed=42):
    set_seed(seed)
    result = train_from_tensors(
        model=model,
        X_train=bundle.X_train,
        y_train=y_train,
        X_val=bundle.X_val,
        y_val=y_val,
        device=device,
        cfg=cfg,
    )
    row = {
        "family": family,
        "task": task,
        "run_name": cfg.run_name,
        "seed": seed,
        "lr": cfg.lr,
        "weight_decay": cfg.weight_decay,
        "batch_size": cfg.batch_size,
        "val_batch_size": cfg.val_batch_size,
        "max_epochs": cfg.max_epochs,
        "patience": cfg.patience,
        "print_every": cfg.print_every,
        "grad_clip_norm": cfg.grad_clip_norm,
        "best_epoch": result.best_epoch,
    }
    row.update(result.best_metrics)
    return row

def task_data(task_name):
    if task_name == "joint":
        return joint_y_train, joint_y_val
    if task_name == "multitask":
        return mt_y_train, mt_y_val
    if task_name == "coordinate":
        return coord_y_train, coord_y_val
    raise ValueError(task_name)

def build_model(family, task_name):
    if family == "cnn":
        if task_name == "joint":
            return CNNJointModel(in_dim=in_dim)
        if task_name == "multitask":
            return CNNMultiTaskModel(in_dim=in_dim)
        if task_name == "coordinate":
            return CNNCoordinateModel(in_dim=in_dim, coordinate_std=bundle.coordinate_std)

    if family == "lstm":
        if task_name == "joint":
            return LSTMJointModel(in_dim=in_dim)
        if task_name == "multitask":
            return LSTMMultiTaskModel(in_dim=in_dim)
        if task_name == "coordinate":
            return LSTMCoordinateModel(in_dim=in_dim, coordinate_std=bundle.coordinate_std)

    raise ValueError((family, task_name))


## Tuning grids

These are the final bounded-tuning grids for CNN and LSTM.

Rules:
- fixed CNN and LSTM architectures
- no channel/hidden-size/layer-count search
- five optimizer/schedule candidates per task and family
- CNN uses the same LR/schedule shape as MLP
- LSTM uses the same number of candidates, but the LR grid is shifted lower and gradient clipping is enabled because long recurrent sequences are less stable
- selection on clean validation score only
- 3-seed confirmation after selection

In [8]:
BASELINE_CFG = dict(
    lr=2e-3,
    weight_decay=1e-4,
    batch_size=256,
    val_batch_size=512,
    max_epochs=30,
    patience=5,
    print_every=5,
    grad_clip_norm=None,
)

# Bounded 5-candidate grids.
# CNN uses the same optimizer/schedule shape as MLP.
# LSTM uses the same number of candidates, but with a lower-LR-biased grid
# and gradient clipping because long recurrent sequences are less stable.
def cnn_specs(prefix: str):
    return [
        dict(run_name=f"{prefix}_base_2e3_wd1e4",        lr=2e-3, weight_decay=1e-4, max_epochs=30, patience=5,  grad_clip_norm=None),
        dict(run_name=f"{prefix}_stable_1e3_wd1e4",     lr=1e-3, weight_decay=1e-4, max_epochs=50, patience=10, grad_clip_norm=None),
        dict(run_name=f"{prefix}_reg_1e3_wd5e4",        lr=1e-3, weight_decay=5e-4, max_epochs=50, patience=10, grad_clip_norm=None),
        dict(run_name=f"{prefix}_low_5e4_wd1e4",        lr=5e-4, weight_decay=1e-4, max_epochs=60, patience=12, grad_clip_norm=None),
        dict(run_name=f"{prefix}_lowreg_5e4_wd5e4",     lr=5e-4, weight_decay=5e-4, max_epochs=80, patience=15, grad_clip_norm=None),
    ]

def lstm_specs(prefix: str):
    return [
        dict(run_name=f"{prefix}_stable_1e3_wd1e4_gc1", lr=1e-3, weight_decay=1e-4, max_epochs=50, patience=10, grad_clip_norm=1.0),
        dict(run_name=f"{prefix}_reg_1e3_wd5e4_gc1",    lr=1e-3, weight_decay=5e-4, max_epochs=50, patience=10, grad_clip_norm=1.0),
        dict(run_name=f"{prefix}_low_5e4_wd1e4_gc1",    lr=5e-4, weight_decay=1e-4, max_epochs=60, patience=12, grad_clip_norm=1.0),
        dict(run_name=f"{prefix}_lowreg_5e4_wd5e4_gc1", lr=5e-4, weight_decay=5e-4, max_epochs=80, patience=15, grad_clip_norm=1.0),
        dict(run_name=f"{prefix}_slow_1e4_wd1e4_gc1",   lr=1e-4, weight_decay=1e-4, max_epochs=80, patience=15, grad_clip_norm=1.0),
    ]

CNN_GRIDS = {
    "joint": cnn_specs("cnn_joint"),
    "multitask": cnn_specs("cnn_mt"),
    "coordinate": cnn_specs("cnn_coord"),
}

LSTM_GRIDS = {
    "joint": lstm_specs("lstm_joint"),
    "multitask": lstm_specs("lstm_mt"),
    "coordinate": lstm_specs("lstm_coord"),
}

BASELINE_CFG, CNN_GRIDS, LSTM_GRIDS

({'lr': 0.002,
  'weight_decay': 0.0001,
  'batch_size': 256,
  'val_batch_size': 512,
  'max_epochs': 30,
  'patience': 5,
  'print_every': 5,
  'grad_clip_norm': None},
 {'joint': [{'run_name': 'cnn_joint_base_2e3_wd1e4',
    'lr': 0.002,
    'weight_decay': 0.0001,
    'max_epochs': 30,
    'patience': 5,
    'grad_clip_norm': None},
   {'run_name': 'cnn_joint_stable_1e3_wd1e4',
    'lr': 0.001,
    'weight_decay': 0.0001,
    'max_epochs': 50,
    'patience': 10,
    'grad_clip_norm': None},
   {'run_name': 'cnn_joint_reg_1e3_wd5e4',
    'lr': 0.001,
    'weight_decay': 0.0005,
    'max_epochs': 50,
    'patience': 10,
    'grad_clip_norm': None},
   {'run_name': 'cnn_joint_low_5e4_wd1e4',
    'lr': 0.0005,
    'weight_decay': 0.0001,
    'max_epochs': 60,
    'patience': 12,
    'grad_clip_norm': None},
   {'run_name': 'cnn_joint_lowreg_5e4_wd5e4',
    'lr': 0.0005,
    'weight_decay': 0.0005,
    'max_epochs': 80,
    'patience': 15,
    'grad_clip_norm': None}],
  'multitask': [

## Run CNN/LSTM tuning


In [9]:
RUN_GRID = True
GRID_SEED = 42

all_results = []

if RUN_GRID:
    for family, grids in [("cnn", CNN_GRIDS), ("lstm", LSTM_GRIDS)]:
        for task_name, trials in grids.items():
            y_train, y_val = task_data(task_name)
            print(f"\n===== {family.upper()} tuning: {task_name} =====")
            for spec in trials:
                print(f"\n--- {spec['run_name']} ---")
                set_seed(GRID_SEED)
                model = build_model(family, task_name)
                cfg = make_cfg(**spec)
                row = run_trial(model, y_train, y_val, cfg, family=family, task=task_name, seed=GRID_SEED)
                row["params"] = count_trainable_params(model)
                all_results.append(row)

cnn_lstm_tuning_df = pd.DataFrame(all_results)
cnn_lstm_tuning_df



===== CNN tuning: joint =====

--- cnn_joint_base_2e3_wd1e4 ---
epoch=001 train_loss=1.3675 val_loss=2.3942 score=0.1800
epoch=005 train_loss=0.2184 val_loss=1.7189 score=0.5365
epoch=010 train_loss=0.0922 val_loss=1.1894 score=0.7075

--- cnn_joint_stable_1e3_wd1e4 ---
epoch=001 train_loss=1.3382 val_loss=1.7680 score=0.3609
epoch=005 train_loss=0.1964 val_loss=1.4613 score=0.5644
epoch=010 train_loss=0.0839 val_loss=0.8666 score=0.7732
epoch=015 train_loss=0.0536 val_loss=1.5993 score=0.6688
epoch=020 train_loss=0.0237 val_loss=0.9289 score=0.8110
epoch=025 train_loss=0.0248 val_loss=0.9586 score=0.8164
epoch=030 train_loss=0.0241 val_loss=0.9051 score=0.8191
epoch=035 train_loss=0.0138 val_loss=0.9664 score=0.8047
epoch=040 train_loss=0.0124 val_loss=1.0185 score=0.8137
epoch=045 train_loss=0.0110 val_loss=0.9995 score=0.8380
epoch=050 train_loss=0.0089 val_loss=1.0697 score=0.8137

--- cnn_joint_reg_1e3_wd5e4 ---
epoch=001 train_loss=1.3341 val_loss=1.7665 score=0.3465
epoch=005 t

,family,task,run_name,seed,lr,weight_decay,batch_size,val_batch_size,max_epochs,patience,print_every,grad_clip_norm,best_epoch,epoch,train_loss,val_loss,score,joint_accuracy,building_accuracy,floor_accuracy,params,coordinate_mean_euclidean,coordinate_rmse,coordinate_mean_euclidean_m,coordinate_rmse_m
0,cnn,joint,cnn_joint_base_2e3_wd1e4,42,0.0020,0.0001,256,512,30,5,5,NaN,9,9.0,0.114045,0.953754,0.754275,0.754275,0.909091,0.783978,1584909,NaN,NaN,NaN,NaN
1,cnn,joint,cnn_joint_stable_1e3_wd1e4,42,0.0010,0.0001,256,512,50,10,5,NaN,45,45.0,0.010952,0.999468,0.837984,0.837984,0.947795,0.851485,1584909,NaN,NaN,NaN,NaN
2,cnn,joint,cnn_joint_reg_1e3_wd5e4,42,0.0010,0.0005,256,512,50,10,5,NaN,33,33.0,0.016187,0.928447,0.840684,0.840684,0.947795,0.852385,1584909,NaN,NaN,NaN,NaN
3,cnn,joint,cnn_joint_low_5e4_wd1e4,42,0.0005,0.0001,256,512,60,12,5,NaN,41,41.0,0.012570,0.901576,0.835284,0.835284,0.952295,0.849685,1584909,NaN,NaN,NaN,NaN
4,cnn,joint,cnn_joint_lowreg_5e4_wd5e4,42,0.0005,0.0005,256,512,80,15,5,NaN,41,41.0,0.014352,0.858755,0.824482,0.824482,0.945095,0.833483,1584909,NaN,NaN,NaN,NaN
5,cnn,multitask,cnn_mt_base_2e3_wd1e4,42,0.0020,0.0001,256,512,30,5,5,NaN,23,23.0,0.073749,1.011967,0.780378,0.780378,0.954995,0.798380,1583624,NaN,NaN,NaN,NaN
6,cnn,multitask,cnn_mt_stable_1e3_wd1e4,42,0.0010,0.0001,256,512,50,10,5,NaN,46,46.0,0.016185,1.240370,0.826283,0.826283,0.970297,0.838884,1583624,NaN,NaN,NaN,NaN
7,cnn,multitask,cnn_mt_reg_1e3_wd5e4,42,0.0010,0.0005,256,512,50,10,5,NaN,30,30.0,0.032554,1.006070,0.818182,0.818182,0.969397,0.827183,1583624,NaN,NaN,NaN,NaN
8,cnn,multitask,cnn_mt_low_5e4_wd1e4,42,0.0005,0.0001,256,512,60,12,5,NaN,29,29.0,0.032316,1.069153,0.810981,0.810981,0.971197,0.827183,1583624,NaN,NaN,NaN,NaN
9,cnn,multitask,cnn_mt_lowreg_5e4_wd5e4,42,0.0005,0.0005,256,512,80,15,5,NaN,47,47.0,0.019753,1.033071,0.817282,0.817282,0.970297,0.826283,1583624,NaN,NaN,NaN,NaN


## Select best candidate per family/task


In [10]:
summary_cols = [
    "family", "task", "run_name", "seed", "best_epoch", "score",
    "joint_accuracy", "building_accuracy", "floor_accuracy",
    "coordinate_mean_euclidean_m", "coordinate_rmse_m",
    "lr", "weight_decay", "max_epochs", "patience", "grad_clip_norm", "params"
]

display(cnn_lstm_tuning_df[[c for c in summary_cols if c in cnn_lstm_tuning_df.columns]].sort_values(["family", "task", "score"], ascending=[True, True, False]))

best_by_family_task = (
    cnn_lstm_tuning_df
    .sort_values(["family", "task", "score"], ascending=[True, True, False])
    .groupby(["family", "task"], as_index=False)
    .first()
)

best_by_family_task


,family,task,run_name,seed,best_epoch,score,joint_accuracy,building_accuracy,floor_accuracy,coordinate_mean_euclidean_m,coordinate_rmse_m,lr,weight_decay,max_epochs,patience,grad_clip_norm,params
13,cnn,coordinate,cnn_coord_low_5e4_wd1e4,42,58,-40.884560,NaN,NaN,NaN,40.884560,58.284143,0.0005,0.0001,60,12,NaN,1582082
14,cnn,coordinate,cnn_coord_lowreg_5e4_wd5e4,42,69,-41.204490,NaN,NaN,NaN,41.204490,59.368368,0.0005,0.0005,80,15,NaN,1582082
12,cnn,coordinate,cnn_coord_reg_1e3_wd5e4,42,45,-41.473849,NaN,NaN,NaN,41.473849,59.586189,0.0010,0.0005,50,10,NaN,1582082
11,cnn,coordinate,cnn_coord_stable_1e3_wd1e4,42,45,-43.904909,NaN,NaN,NaN,43.904909,61.645744,0.0010,0.0001,50,10,NaN,1582082
10,cnn,coordinate,cnn_coord_base_2e3_wd1e4,42,23,-53.989826,NaN,NaN,NaN,53.989826,71.196367,0.0020,0.0001,30,5,NaN,1582082
2,cnn,joint,cnn_joint_reg_1e3_wd5e4,42,33,0.840684,0.840684,0.947795,0.852385,NaN,NaN,0.0010,0.0005,50,10,NaN,1584909
1,cnn,joint,cnn_joint_stable_1e3_wd1e4,42,45,0.837984,0.837984,0.947795,0.851485,NaN,NaN,0.0010,0.0001,50,10,NaN,1584909
3,cnn,joint,cnn_joint_low_5e4_wd1e4,42,41,0.835284,0.835284,0.952295,0.849685,NaN,NaN,0.0005,0.0001,60,12,NaN,1584909
4,cnn,joint,cnn_joint_lowreg_5e4_wd5e4,42,41,0.824482,0.824482,0.945095,0.833483,NaN,NaN,0.0005,0.0005,80,15,NaN,1584909
0,cnn,joint,cnn_joint_base_2e3_wd1e4,42,9,0.754275,0.754275,0.909091,0.783978,NaN,NaN,0.0020,0.0001,30,5,NaN,1584909


,family,task,run_name,seed,lr,weight_decay,batch_size,val_batch_size,max_epochs,patience,print_every,grad_clip_norm,best_epoch,epoch,train_loss,val_loss,score,joint_accuracy,building_accuracy,floor_accuracy,params,coordinate_mean_euclidean,coordinate_rmse,coordinate_mean_euclidean_m,coordinate_rmse_m
0,cnn,coordinate,cnn_coord_low_5e4_wd1e4,42,0.0005,0.0001,256,512,60,12,5,NaN,58,58.0,0.042026,0.183814,-40.884560,NaN,NaN,NaN,1582082,0.426828,0.598673,40.884560,58.284143
1,cnn,joint,cnn_joint_reg_1e3_wd5e4,42,0.0010,0.0005,256,512,50,10,5,NaN,33,33.0,0.016187,0.928447,0.840684,0.840684,0.947795,0.852385,1584909,NaN,NaN,NaN,NaN
2,cnn,multitask,cnn_mt_stable_1e3_wd1e4,42,0.0010,0.0001,256,512,50,10,5,NaN,46,46.0,0.016185,1.240370,0.826283,0.826283,0.970297,0.838884,1583624,NaN,NaN,NaN,NaN
3,lstm,coordinate,lstm_coord_stable_1e3_wd1e4_gc1,42,0.0010,0.0001,256,512,50,10,5,1.0,36,36.0,0.018486,0.014667,-10.574129,NaN,NaN,NaN,1639203,0.119681,0.165398,10.574129,14.367968
4,lstm,joint,lstm_joint_lowreg_5e4_wd5e4_gc1,42,0.0005,0.0005,256,512,80,15,5,1.0,17,17.0,0.014419,0.259721,0.953195,0.953195,0.998200,0.953195,1642030,NaN,NaN,NaN,NaN
5,lstm,multitask,lstm_mt_low_5e4_wd1e4_gc1,42,0.0005,0.0001,256,512,60,12,5,1.0,24,24.0,0.012488,0.337477,0.951395,0.951395,0.998200,0.951395,1640745,NaN,NaN,NaN,NaN


## 3-seed confirmation for selected CNN/LSTM configs


In [11]:
RUN_CONFIRM_FINAL = True
SEEDS_CONFIRM = (42, 123, 999)

confirm_rows = []

if RUN_CONFIRM_FINAL:
    for _, best in best_by_family_task.iterrows():
        family = best["family"]
        task_name = best["task"]
        y_train, y_val = task_data(task_name)

        spec = {
            "run_name": f"confirm_{best['run_name']}",
            "lr": float(best["lr"]),
            "weight_decay": float(best["weight_decay"]),
            "max_epochs": int(best["max_epochs"]),
            "patience": int(best["patience"]),
            "print_every": 5,
            "grad_clip_norm": None if pd.isna(best.get("grad_clip_norm", None)) else float(best["grad_clip_norm"]),
        }

        for seed in SEEDS_CONFIRM:
            print(f"\n===== Confirm {family.upper()} {task_name}, seed={seed} =====")
            set_seed(seed)
            model = build_model(family, task_name)
            cfg = make_cfg(**spec)
            row = run_trial(model, y_train, y_val, cfg, family=family, task=task_name, seed=seed)
            row["selected_from_run"] = best["run_name"]
            row["params"] = count_trainable_params(model)
            confirm_rows.append(row)

cnn_lstm_confirm_runs_df = pd.DataFrame(confirm_rows)
cnn_lstm_confirm_runs_df



===== Confirm CNN coordinate, seed=42 =====
epoch=001 train_loss=0.5410 val_loss=0.7314 score=-108.1132
epoch=005 train_loss=0.2260 val_loss=0.4723 score=-80.6915
epoch=010 train_loss=0.1480 val_loss=0.3960 score=-73.2173
epoch=015 train_loss=0.1164 val_loss=0.3224 score=-63.5129
epoch=020 train_loss=0.0968 val_loss=0.5092 score=-77.9589
epoch=025 train_loss=0.0722 val_loss=0.2330 score=-49.6531
epoch=030 train_loss=0.0673 val_loss=0.2561 score=-50.7101
epoch=035 train_loss=0.0633 val_loss=0.3845 score=-64.2121
epoch=040 train_loss=0.0597 val_loss=0.2771 score=-58.5533
epoch=045 train_loss=0.0496 val_loss=0.1907 score=-42.3866
epoch=050 train_loss=0.0463 val_loss=0.2243 score=-47.1859
epoch=055 train_loss=0.0435 val_loss=0.2160 score=-46.1125
epoch=060 train_loss=0.0410 val_loss=0.1828 score=-41.1664

===== Confirm CNN coordinate, seed=123 =====
epoch=001 train_loss=0.5460 val_loss=1.1679 score=-118.9436
epoch=005 train_loss=0.2311 val_loss=0.5044 score=-81.3395
epoch=010 train_loss=0

,family,task,run_name,seed,lr,weight_decay,batch_size,val_batch_size,max_epochs,patience,print_every,grad_clip_norm,best_epoch,epoch,train_loss,val_loss,score,coordinate_mean_euclidean,coordinate_rmse,coordinate_mean_euclidean_m,coordinate_rmse_m,selected_from_run,params,joint_accuracy,building_accuracy,floor_accuracy
0,cnn,coordinate,confirm_cnn_coord_low_5e4_wd1e4,42,0.0005,0.0001,256,512,60,12,5,NaN,58,58.0,0.042026,0.183814,-40.884560,0.426828,0.598673,40.884560,58.284143,cnn_coord_low_5e4_wd1e4,1582082,NaN,NaN,NaN
1,cnn,coordinate,confirm_cnn_coord_low_5e4_wd1e4,123,0.0005,0.0001,256,512,60,12,5,NaN,58,58.0,0.051192,0.194471,-42.437875,0.446286,0.618183,42.437875,60.054152,cnn_coord_low_5e4_wd1e4,1582082,NaN,NaN,NaN
2,cnn,coordinate,confirm_cnn_coord_low_5e4_wd1e4,999,0.0005,0.0001,256,512,60,12,5,NaN,49,49.0,0.044850,0.181222,-41.214641,0.427890,0.592590,41.214641,58.104056,cnn_coord_low_5e4_wd1e4,1582082,NaN,NaN,NaN
3,cnn,joint,confirm_cnn_joint_reg_1e3_wd5e4,42,0.0010,0.0005,256,512,50,10,5,NaN,33,33.0,0.016187,0.928447,0.840684,NaN,NaN,NaN,NaN,cnn_joint_reg_1e3_wd5e4,1584909,0.840684,0.947795,0.852385
4,cnn,joint,confirm_cnn_joint_reg_1e3_wd5e4,123,0.0010,0.0005,256,512,50,10,5,NaN,32,32.0,0.019822,0.891334,0.828083,NaN,NaN,NaN,NaN,cnn_joint_reg_1e3_wd5e4,1584909,0.828083,0.945995,0.840684
5,cnn,joint,confirm_cnn_joint_reg_1e3_wd5e4,999,0.0010,0.0005,256,512,50,10,5,NaN,28,28.0,0.015543,0.824288,0.828983,NaN,NaN,NaN,NaN,cnn_joint_reg_1e3_wd5e4,1584909,0.828983,0.942394,0.845185
6,cnn,multitask,confirm_cnn_mt_stable_1e3_wd1e4,42,0.0010,0.0001,256,512,50,10,5,NaN,46,46.0,0.016185,1.240370,0.826283,NaN,NaN,NaN,NaN,cnn_mt_stable_1e3_wd1e4,1583624,0.826283,0.970297,0.838884
7,cnn,multitask,confirm_cnn_mt_stable_1e3_wd1e4,123,0.0010,0.0001,256,512,50,10,5,NaN,32,32.0,0.028178,1.140643,0.810081,NaN,NaN,NaN,NaN,cnn_mt_stable_1e3_wd1e4,1583624,0.810081,0.972097,0.821782
8,cnn,multitask,confirm_cnn_mt_stable_1e3_wd1e4,999,0.0010,0.0001,256,512,50,10,5,NaN,35,35.0,0.023363,1.135411,0.818182,NaN,NaN,NaN,NaN,cnn_mt_stable_1e3_wd1e4,1583624,0.818182,0.967597,0.831683
9,lstm,coordinate,confirm_lstm_coord_stable_1e3_wd1e4_gc1,42,0.0010,0.0001,256,512,50,10,5,1.0,36,36.0,0.018486,0.014667,-10.574129,0.119681,0.165398,10.574129,14.367968,lstm_coord_stable_1e3_wd1e4_gc1,1639203,NaN,NaN,NaN


## Confirmation summary


In [12]:
def summarize_confirm(df):
    if df.empty:
        return pd.DataFrame()
    agg = {
        "seed": "count",
        "score": ["mean", "std"],
        "best_epoch": "mean",
        "train_loss": "mean",
        "val_loss": "mean",
        "joint_accuracy": "mean",
        "building_accuracy": "mean",
        "floor_accuracy": "mean",
        "coordinate_mean_euclidean_m": "mean",
        "coordinate_rmse_m": "mean",
        "params": "first",
    }
    available_agg = {k: v for k, v in agg.items() if k in df.columns}
    out = df.groupby(["family", "task", "run_name"]).agg(available_agg)
    out.columns = ["_".join(col).strip("_") if isinstance(col, tuple) else col for col in out.columns]
    out = out.reset_index().rename(columns={"seed_count": "runs"})
    return out

cnn_lstm_confirmation_summary = summarize_confirm(cnn_lstm_confirm_runs_df)
cnn_lstm_confirmation_summary


,family,task,run_name,runs,score_mean,score_std,best_epoch_mean,train_loss_mean,val_loss_mean,joint_accuracy_mean,building_accuracy_mean,floor_accuracy_mean,coordinate_mean_euclidean_m_mean,coordinate_rmse_m_mean,params_first
0,cnn,coordinate,confirm_cnn_coord_low_5e4_wd1e4,3,-41.512358,0.818336,55.000000,0.046023,0.186502,NaN,NaN,NaN,41.512358,58.814117,1582082
1,cnn,joint,confirm_cnn_joint_reg_1e3_wd5e4,3,0.832583,0.007030,31.000000,0.017184,0.881356,0.832583,0.945395,0.846085,NaN,NaN,1584909
2,cnn,multitask,confirm_cnn_mt_stable_1e3_wd1e4,3,0.818182,0.008101,37.666667,0.022575,1.172141,0.818182,0.969997,0.830783,NaN,NaN,1583624
3,lstm,coordinate,confirm_lstm_coord_stable_1e3_wd1e4_gc1,3,-10.875420,0.321031,36.666667,0.019091,0.016238,NaN,NaN,NaN,10.875420,15.566495,1639203
4,lstm,joint,confirm_lstm_joint_lowreg_5e4_wd5e4_gc1,3,0.945995,0.014872,27.333333,0.010415,0.366399,0.945995,0.998200,0.945995,NaN,NaN,1642030
5,lstm,multitask,confirm_lstm_mt_low_5e4_wd1e4_gc1,3,0.946295,0.004440,25.333333,0.013809,0.376508,0.946295,0.998200,0.946595,NaN,NaN,1640745


## Final config dictionary to copy into baseline/robustness notebooks


In [13]:
FINAL_TUNED_CFGS = {
    family: {
        row["task"]: {
            "lr": float(row["lr"]),
            "weight_decay": float(row["weight_decay"]),
            "grad_clip_norm": None if pd.isna(row.get("grad_clip_norm", None)) else float(row["grad_clip_norm"]),
            "max_epochs": int(row["max_epochs"]),
            "patience": int(row["patience"]),
            "print_every": 5,
            "batch_size": 256,
            "val_batch_size": 512,
        }
        for _, row in g.iterrows()
    }
    for family, g in best_by_family_task.groupby("family")
}

FINAL_TUNED_CFGS


{'cnn': {'coordinate': {'lr': 0.0005,
   'weight_decay': 0.0001,
   'grad_clip_norm': None,
   'max_epochs': 60,
   'patience': 12,
   'print_every': 5,
   'batch_size': 256,
   'val_batch_size': 512},
  'joint': {'lr': 0.001,
   'weight_decay': 0.0005,
   'grad_clip_norm': None,
   'max_epochs': 50,
   'patience': 10,
   'print_every': 5,
   'batch_size': 256,
   'val_batch_size': 512},
  'multitask': {'lr': 0.001,
   'weight_decay': 0.0001,
   'grad_clip_norm': None,
   'max_epochs': 50,
   'patience': 10,
   'print_every': 5,
   'batch_size': 256,
   'val_batch_size': 512}},
 'lstm': {'coordinate': {'lr': 0.001,
   'weight_decay': 0.0001,
   'grad_clip_norm': 1.0,
   'max_epochs': 50,
   'patience': 10,
   'print_every': 5,
   'batch_size': 256,
   'val_batch_size': 512},
  'joint': {'lr': 0.0005,
   'weight_decay': 0.0005,
   'grad_clip_norm': 1.0,
   'max_epochs': 80,
   'patience': 15,
   'print_every': 5,
   'batch_size': 256,
   'val_batch_size': 512},
  'multitask': {'lr': 0.0

## Save outputs


In [14]:
OUTPUT_DIR = PROJECT_ROOT / "notebooks" / "logs" / "fair_tuning"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not cnn_lstm_tuning_df.empty:
    cnn_lstm_tuning_df.to_csv(OUTPUT_DIR / "cnn_lstm_tuning_grid_results.csv", index=False)
if not cnn_lstm_confirm_runs_df.empty:
    cnn_lstm_confirm_runs_df.to_csv(OUTPUT_DIR / "cnn_lstm_confirm_runs.csv", index=False)
if not cnn_lstm_confirmation_summary.empty:
    cnn_lstm_confirmation_summary.to_csv(OUTPUT_DIR / "cnn_lstm_confirmation_summary.csv", index=False)

print("Saved outputs to:", OUTPUT_DIR)


Saved outputs to: /storage/ice1/7/5/kw53/cs7643/tmp/CS7643-Project/notebooks/logs/fair_tuning
